# Phase 2 SFT (Supervised Fine-Tuning) for Multimodal Lily

This notebook performs instruction tuning on **Lily** using the multimodal dataset `lmms-lab/multimodal-open-r1-8k-verified` and text reasoning samples from `abhinav0231/Sarvam-105b-Distill-100k`.

**Key Features**:
1. **LoRA config**: LLM is optimized using Unsloth LoRA kernels ($r=32$, $\alpha=64$).
2. **Projector**: Trainable 2-layer MLP loaded with Phase 1 alignment weights.
3. **Vision Tower**: SigLIP-2 (frozen, extracting features from the second-to-last layer).
4. **Weight Checkpointing & Resume**: Projector + LoRA adapters + Optimizer + Scheduler weights are checkpointed every 150 steps and pushed to Hugging Face Hub. Resuming is fully automated.
5. **WandB Logging**: Loss and learning rate logged every 5 steps.
6. **Adapter Merge & Push**: Lossless 16-bit precision merge of LLM and LoRA adapters, pushed to a new repo along with final projector weights.

In [1]:
# Configure Hub acceleration before importing huggingface_hub / datasets.
# Use the official endpoint unless HF_ENDPOINT is explicitly set by the runtime.
import os
os.environ.setdefault("HF_ENDPOINT", "https://huggingface.co")
os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")

# Install optimized dependencies
%uv pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%uv pip install -q wandb datasets "huggingface_hub[hf_xet]" accelerate

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import random
import re
import glob
import shutil
import time
import warnings
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset as TorchDataset, Sampler
from torch.utils.data import DataLoader

import wandb
import transformers
from transformers import (
    AutoModel,
    AutoModelForCausalLM,
    AutoTokenizer,
    AutoProcessor,
    get_cosine_schedule_with_warmup,
)
from transformers.utils.hub import PushToHubMixin
from datasets import load_dataset
from accelerate import Accelerator
from huggingface_hub import login, HfApi, snapshot_download, hf_hub_download, upload_folder
from safetensors.torch import load_file

from unsloth import FastLanguageModel
from peft import PeftModel, set_peft_model_state_dict
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

print(f"Torch: {torch.__version__}")
print(f"CUDA:  {torch.version.cuda}")
print(f"Devices: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}  {p.total_memory/1e9:.0f}GB  cc={p.major}.{p.minor}")

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

IS_BF16_SUPPORTED = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
DTYPE = torch.bfloat16 if IS_BF16_SUPPORTED else torch.float16
print(f"Selected precision: {DTYPE}")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.8.0+cu129).
/usr/local/lib/python3.12/site-packages/torchao/quantization/quant_api.py:1731: SyntaxWarning: invalid escape sequence '\.'
  """Configuration class for applying different quantization configs to modules or parameters based on their fully qualified names (FQNs).
/usr/local/lib/python3.12/site-packages/unsloth/__init__.py:1432: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


[bitsandbytes.cextension|WARNING]No prebuilt binary for CUDA 12.9, loading CUDA 12.8 instead. Set BNB_CUDA_VERSION to override.


🦥 Unsloth Zoo will now patch everything to make training faster!
Torch: 2.8.0+cu129
CUDA:  12.9
Devices: 1
GPU 0: NVIDIA A100-SXM4-40GB  42GB  cc=8.0
Selected precision: torch.bfloat16


In [2]:
# Hub endpoint and Xet acceleration were configured before Hub imports.
# Do not force hf-mirror here: it is often slower from Modal's US regions.

HF_USERNAME = "abhinav0231"

# Models
LLM_MODEL_ID = "abhinav0231/Lily-1.5b-v0.3"
VISION_MODEL_ID = "google/siglip2-so400m-patch14-384"
PROJECTOR_PRETRAINED_REPO = "abhinav0231/Lily-1.5b-projector-siglip2"

# Hyperparameters
MAX_SEQ_LENGTH = 3072
LORA_RANK = 32
LORA_ALPHA = 64
SEED = 42

BATCH_SIZE = 24
GRAD_ACCUM_STEPS = 2     # Effective batch size = 48
USE_GRADIENT_CHECKPOINTING = True  # Standard GPU recomputation (disables slow CPU H2D RAM offloading)
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03
NUM_EPOCHS = 1

# Checkpointing & Logging
SAVE_STEPS = 250
LOGGING_STEPS = 10

CHECKPOINT_REPO = f"{HF_USERNAME}/Lily-1.5b-SFT-checkpoints"
HF_MERGED_REPO = f"{HF_USERNAME}/Lily-1.5b-SFT-siglip2"
RESUME_FROM_CHECKPOINT = True

# Paths
OUTPUT_DIR = "./sft_output"
MERGED_DIR = "./sft_merged"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
# On Modal, inject secrets via environment or fetch them safely
HF_TOKEN = os.environ.get("HF_TOKEN")
WANDB_API_KEY = os.environ.get("WANDB_API_KEY")

login(token=HF_TOKEN, add_to_git_credential=False)
wandb.login(key=WANDB_API_KEY, relogin=True)

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["WANDB_API_KEY"] = WANDB_API_KEY
print(">> Authenticated successfully with Hugging Face & WandB")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: abhinav0231 (abhinav0231-krmangalam) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


>> Authenticated successfully with Hugging Face & WandB


In [4]:
class LilyVLM(nn.Module):
    def __init__(self, vision_model_id, llm_model_id, tokenizer, max_seq_length=2048, lora_rank=32, lora_alpha=64, seed=42):
        super().__init__()
        self.tokenizer = tokenizer
        
        print("Loading vision tower (SigLIP-2)...")
        self.vision_tower = AutoModel.from_pretrained(vision_model_id, torch_dtype=DTYPE)
        if hasattr(self.vision_tower, "vision_model"):
            self.vision_tower = self.vision_tower.vision_model

        # Freeze vision tower to save VRAM and keep visual features stable
        for param in self.vision_tower.parameters():
            param.requires_grad = False
            
        print("Loading LLM with Unsloth fast kernels...")
        self.language_model, _ = FastLanguageModel.from_pretrained(
            model_name     = llm_model_id,
            max_seq_length = max_seq_length,
            dtype          = DTYPE,
            load_in_4bit   = True,
        )
        self.language_model.resize_token_embeddings(len(tokenizer))
        
        # Wrap language model with LoRA using Unsloth
        self.language_model = FastLanguageModel.get_peft_model(
            self.language_model,
            r                          = lora_rank,
            lora_alpha                 = lora_alpha,
            target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                          "gate_proj", "up_proj", "down_proj"],
            lora_dropout               = 0,
            bias                       = "none",
            # GPU-native gradient checkpointing (no CPU RAM offloading bottleneck)
            use_gradient_checkpointing = True if USE_GRADIENT_CHECKPOINTING else False,
            random_state               = seed,
        )

        # MLP Projector mapping SigLIP-2 (1152) -> Lily LLM (1536)
        in_dim = self.vision_tower.config.hidden_size
        llm_dim = self.language_model.config.hidden_size
        self.projector = nn.Sequential(
            nn.Linear(in_dim, 2048),
            nn.SiLU(),
            nn.Linear(2048, llm_dim)
        )
        
        # UNFREEZE MLP Projector for joint fine-tuning with LoRA
        print("Unfreezing MLP Projector for joint fine-tuning with LoRA...")
        for param in self.projector.parameters():
            param.requires_grad = True

        self.image_token_id = tokenizer.convert_tokens_to_ids("<image>")

        self._penultimate_features = None
        vision_encoder = getattr(self.vision_tower, "encoder", None)
        vision_layers = getattr(vision_encoder, "layers", None)
        if vision_layers is None or len(vision_layers) < 2:
            raise RuntimeError("Could not locate SigLIP-2 encoder layers for penultimate feature extraction.")

        vision_layers[-2].register_forward_hook(self._capture_penultimate_layer)

    def _capture_penultimate_layer(self, _module, _inputs, output):
        self._penultimate_features = output[0] if isinstance(output, (tuple, list)) else output

    def train(self, mode=True):
        super().train(mode)
        # The frozen tower must remain deterministic and should not run dropout/drop-path.
        self.vision_tower.eval()
        return self

    def forward(self, input_ids, attention_mask, labels, pixel_values):
        inputs_embeds = self.language_model.get_input_embeddings()(input_ids).clone()
        image_token_mask = (input_ids == self.image_token_id)
        
        if pixel_values.shape[0] > 0:
            self._penultimate_features = None
            with torch.no_grad():
                self.vision_tower(pixel_values, return_dict=True)
            image_features = self._penultimate_features
            if not isinstance(image_features, torch.Tensor):
                raise RuntimeError("Failed to capture SigLIP-2's penultimate encoder layer.")

            # Full 729 visual tokens (27x27 grid at 384x384 resolution) passed directly to projector!
            image_embeddings = self.projector(image_features) # Shape: [B, 729, 1536]

            flat_image_embeds = image_embeddings.flatten(0, 1).to(inputs_embeds.dtype)
            inputs_embeds[image_token_mask] = flat_image_embeds
        
        outputs = self.language_model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels,
            return_dict=True
        )
        return outputs

In [5]:
from PIL import Image
from huggingface_hub import snapshot_download
from datasets import load_dataset
import os

Image.MAX_IMAGE_PIXELS = None

def load_and_prepare_data(hf_token):
    target_repo = "abhinav0231/Lily-Vision-SFT-Dataset"
    print(f">> Downloading packaged dataset from Hugging Face with 16 parallel workers: {target_repo}...")
    try:
        # snapshot_download downloads all 58 parquet shards concurrently across parallel worker threads
        workers = min(16, (os.cpu_count() or 4) * 2)
        local_data_dir = snapshot_download(
            repo_id=target_repo,
            repo_type="dataset",
            token=hf_token,
            max_workers=workers,
            resume_download=True,
        )
        print(f">> Dataset files cached locally at {local_data_dir}. Loading dataset...")
        ds = load_dataset("parquet", data_files=os.path.join(local_data_dir, "data", "*.parquet"), split="train")
        print(f"Success! Loaded dataset metadata with {len(ds):,} samples.")
        return ds
    except Exception as e:
        print(f"WARNING: Parallel snapshot download failed ({e}), falling back to direct load_dataset...")
        ds = load_dataset(target_repo, split="train", token=hf_token)
        print(f"Success! Loaded dataset metadata with {len(ds):,} samples.")
        return ds

train_ds = load_and_prepare_data(HF_TOKEN)

>> Downloading packaged dataset from Hugging Face with 16 parallel workers: abhinav0231/Lily-Vision-SFT-Dataset...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 60 files:   0%|          | 0/60 [00:00<?, ?it/s]

>> Dataset files cached locally at /root/.cache/huggingface/hub/datasets--abhinav0231--Lily-Vision-SFT-Dataset/snapshots/a743948be3d0ac0f379e9a0ef3cb10df245f9be5. Loading dataset...


Resolving data files:   0%|          | 0/58 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/75 [00:00<?, ?it/s]

Success! Loaded dataset metadata with 64,000 samples.


In [6]:
class MixedSFTDataset(TorchDataset):
    def __init__(self, samples, tokenizer, processor, max_length=2048):
        self.samples = samples
        self.tokenizer = tokenizer
        self.processor = processor
        self.max_length = max_length
        # Approximate length to group samples of similar length into the same batch,
        # eliminating wasteful padding compute (reduces training time from 3+ hours to ~20 mins).
        self.lengths = [
            min(max_length, (len(prompt) + len(response)) // 4 + 2 + (324 if "<image>" in prompt else 0))
            for prompt, response in zip(samples["prompt"], samples["response"])
        ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        
        prompt = item["prompt"]
        response = item["response"]

        prompt_tokens = self.tokenizer.encode(prompt, add_special_tokens=False)
        response_tokens = self.tokenizer.encode(response, add_special_tokens=False)

        # Expand the single <image> placeholder token to 324 tokens on the CPU
        image_token_id = self.tokenizer.convert_tokens_to_ids("<image>")
        has_image = image_token_id in prompt_tokens
        if has_image:
            img_idx = prompt_tokens.index(image_token_id)
            prompt_tokens = prompt_tokens[:img_idx] + [image_token_id] * 729 + prompt_tokens[img_idx+1:]

        # 16k text-only examples have no image placeholder. Avoid both image decoding and SigLIP preprocessing.
        pixel_values = None
        if has_image:
            image_obj = item.get("image", None)
            image_path = item.get("image_path", None)
            if image_obj is not None:
                image = image_obj.convert("RGB")
            elif image_path is not None:
                try:
                    image = Image.open(image_path).convert("RGB")
                except Exception:
                    image = Image.new("RGB", (384, 384), (0, 0, 0))
            else:
                image = Image.new("RGB", (384, 384), (0, 0, 0))

            try:
                pixel_values = self.processor(images=image, return_tensors="pt")["pixel_values"].squeeze(0)
            except Exception:
                dummy = Image.new("RGB", (384, 384), (0, 0, 0))
                pixel_values = self.processor(images=dummy, return_tensors="pt")["pixel_values"].squeeze(0)

        bos_id = self.tokenizer.bos_token_id
        eos_id = self.tokenizer.eos_token_id
        
        bos_tokens = [bos_id] if bos_id is not None else []
        eos_tokens = [eos_id] if eos_id is not None else []

        input_ids = bos_tokens + prompt_tokens + response_tokens + eos_tokens
        labels = [-100] * (len(bos_tokens) + len(prompt_tokens)) + response_tokens + eos_tokens
        attention_mask = [1] * len(input_ids)

        if len(input_ids) > self.max_length:
            input_ids = input_ids[:self.max_length]
            labels = labels[:self.max_length]
            attention_mask = attention_mask[:self.max_length]

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "pixel_values": pixel_values
        }

def collate_fn(batch, pad_token_id):
    input_ids = pad_sequence([item["input_ids"] for item in batch], batch_first=True, padding_value=pad_token_id)
    attention_mask = pad_sequence([item["attention_mask"] for item in batch], batch_first=True, padding_value=0)
    labels = pad_sequence([item["labels"] for item in batch], batch_first=True, padding_value=-100)

    # Tensor-core-friendly sequence widths also reduce the number of distinct shapes.
    padding = (-input_ids.shape[1]) % 8
    if padding:
        input_ids = F.pad(input_ids, (0, padding), value=pad_token_id)
        attention_mask = F.pad(attention_mask, (0, padding), value=0)
        labels = F.pad(labels, (0, padding), value=-100)

    # The order of this stack matches the rows selected by image_token_mask in LilyVLM.forward.
    image_tensors = [item["pixel_values"] for item in batch if item["pixel_values"] is not None]
    pixel_values = torch.stack(image_tensors, dim=0) if image_tensors else torch.empty((0, 3, 384, 384), dtype=torch.float32)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "pixel_values": pixel_values
    }

class LengthGroupedBatchSampler(Sampler):
    """Group batches by similar sequence lengths to eliminate padding waste."""
    def __init__(self, lengths, batch_size, seed=42, mega_batch_mult=50):
        self.lengths = lengths
        self.batch_size = batch_size
        self.seed = seed
        self.mega_batch_mult = mega_batch_mult
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return (len(self.lengths) + self.batch_size - 1) // self.batch_size

    def __iter__(self):
        generator = torch.Generator()
        generator.manual_seed(self.seed + self.epoch)
        indices = torch.randperm(len(self.lengths), generator=generator).tolist()
        mega_batch_size = self.batch_size * self.mega_batch_mult
        batches = []
        for start in range(0, len(indices), mega_batch_size):
            chunk = sorted(indices[start:start + mega_batch_size], key=self.lengths.__getitem__)
            batches.extend(chunk[i:i + self.batch_size] for i in range(0, len(chunk), self.batch_size))

        for batch_idx in torch.randperm(len(batches), generator=generator).tolist():
            yield batches[batch_idx]

In [7]:
_resume_ckpt_dir = None

if RESUME_FROM_CHECKPOINT and CHECKPOINT_REPO:
    try:
        api = HfApi()
        # Create checkpoint repo if not exists
        api.create_repo(CHECKPOINT_REPO, token=HF_TOKEN, exist_ok=True, private=True)
        files = list(api.list_repo_files(CHECKPOINT_REPO, token=HF_TOKEN))
        nums = set()
        for f in files:
            seg = f.split("/")[0]
            if seg.startswith("checkpoint-") and seg.split("-")[-1].isdigit():
                nums.add(int(seg.split("-")[-1]))
        if nums:
            latest = max(nums)
            local_dir = os.path.join(OUTPUT_DIR, f"checkpoint-{latest}")
            print(f"Found checkpoint-{latest} - downloading...")
            snapshot_download(
                repo_id=CHECKPOINT_REPO,
                allow_patterns=[f"checkpoint-{latest}/*"],
                local_dir=OUTPUT_DIR,
                token=HF_TOKEN,
            )
            _resume_ckpt_dir = local_dir
            print(f"Checkpoint downloaded -> {local_dir}")
        else:
            print("No checkpoints found in repo - starting fresh.")
    except Exception as e:
        print(f"WARNING: Resume failed ({e}) - starting fresh.")
print(f"resume_ckpt_dir = {_resume_ckpt_dir}")

No checkpoints found in repo - starting fresh.
resume_ckpt_dir = None


In [8]:
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
if "<image>" not in tokenizer.get_vocab():
    tokenizer.add_tokens(["<image>"], special_tokens=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
    
processor = AutoProcessor.from_pretrained(VISION_MODEL_ID, use_fast=True)

# Initialize VLM model
model = LilyVLM(
    VISION_MODEL_ID, 
    LLM_MODEL_ID, 
    tokenizer, 
    max_seq_length=MAX_SEQ_LENGTH, 
    lora_rank=LORA_RANK, 
    lora_alpha=LORA_ALPHA, 
    seed=SEED
)

# Load Projector Weights
if _resume_ckpt_dir and os.path.exists(os.path.join(_resume_ckpt_dir, "mm_projector.bin")):
    print(f">> Resuming projector weights from checkpoint: {_resume_ckpt_dir}")
    model.projector.load_state_dict(torch.load(os.path.join(_resume_ckpt_dir, "mm_projector.bin"), map_location="cpu"))
else:
    print(">> Downloading and loading Phase 1 pre-trained projector weights...")
    projector_weights_path = hf_hub_download(
        repo_id=PROJECTOR_PRETRAINED_REPO, 
        filename="mm_projector_final.bin",
        token=HF_TOKEN
    )
    model.projector.load_state_dict(torch.load(projector_weights_path, map_location="cpu"))
    print("Projector weights loaded!")

# Load LLM LoRA weights if resuming
if _resume_ckpt_dir:
    print(f">> Resuming LLM LoRA weights from checkpoint: {_resume_ckpt_dir}")
    adapter_model_path = os.path.join(_resume_ckpt_dir, "adapter_model.safetensors")
    if os.path.exists(adapter_model_path):
        adapters_weights = load_file(adapter_model_path)
    else:
        adapters_weights = torch.load(os.path.join(_resume_ckpt_dir, "adapter_model.bin"), map_location="cpu")
        
    set_peft_model_state_dict(model.language_model, adapters_weights)
    print("LoRA adapter weights loaded!")


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.21k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Unsloth 2026.7.5 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unfreezing MLP Projector for joint fine-tuning with LoRA...
>> Downloading and loading Phase 1 pre-trained projector weights...


mm_projector_final.bin: reconstructing file:   0%|          |  0.00B / 22.0MB            

mm_projector_final.bin: downloading bytes:           |  0.00B            

Projector weights loaded!


In [9]:
mixed_precision = "bf16" if IS_BF16_SUPPORTED else "fp16"
accelerator = Accelerator(
    mixed_precision=mixed_precision,
    log_with="wandb",
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
)

num_workers = min(8, os.cpu_count() or 2)
print(f"Configuring DataLoader with {num_workers} parallel workers and length-grouped batching...")
train_dataset = MixedSFTDataset(
    train_ds, tokenizer, processor, max_length=MAX_SEQ_LENGTH
)

batch_sampler = LengthGroupedBatchSampler(train_dataset.lengths, batch_size=BATCH_SIZE, seed=SEED)

loader_kwargs = dict(
    batch_sampler=batch_sampler,
    collate_fn=lambda b: collate_fn(b, tokenizer.pad_token_id),
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=num_workers > 0,
)
if num_workers > 0:
    loader_kwargs["prefetch_factor"] = 4

dataloader = DataLoader(train_dataset, **loader_kwargs)
print(f"DataLoader batches: {len(dataloader)}")  # Prints 1334

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    betas=(0.9, 0.95),
    weight_decay=WEIGHT_DECAY,
)

num_training_steps = (len(dataloader) // GRAD_ACCUM_STEPS) * NUM_EPOCHS
num_warmup_steps = int(num_training_steps * WARMUP_RATIO)

lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

model, optimizer, dataloader, lr_scheduler = accelerator.prepare(
    model, optimizer, dataloader, lr_scheduler
)

if _resume_ckpt_dir:
    opt_path = os.path.join(_resume_ckpt_dir, "optimizer.bin")
    sch_path = os.path.join(_resume_ckpt_dir, "scheduler.bin")
    if os.path.exists(opt_path) and os.path.exists(sch_path):
        optimizer.load_state_dict(torch.load(opt_path, map_location="cpu"))
        lr_scheduler.load_state_dict(torch.load(sch_path, map_location="cpu"))

accelerator.init_trackers(
    project_name="lily-vision-phase2-sft",
    config={
        "learning_rate": LEARNING_RATE,
        "epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    },
)

[accelerate.utils.other|WARNING]Detected kernel version 4.19.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Configuring DataLoader with 8 parallel workers and length-grouped batching...
DataLoader batches: 2667


wandb: setting up run uodc4xzg
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /root/wandb/run-20260727_094711-uodc4xzg
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run charmed-resonance-27
wandb: ⭐️ View project at https://wandb.ai/abhinav0231-krmangalam/lily-vision-phase2-sft
wandb: 🚀 View run at https://wandb.ai/abhinav0231-krmangalam/lily-vision-phase2-sft/runs/uodc4xzg
wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


In [10]:
from tqdm.auto import tqdm

def save_and_push_checkpoint(model, optimizer, lr_scheduler, step, output_dir, repo_id, token):
    ckpt_dir = os.path.join(output_dir, f"checkpoint-{step}")
    os.makedirs(ckpt_dir, exist_ok=True)

    # Save projector weights
    unwrapped = accelerator.unwrap_model(model)
    torch.save(unwrapped.projector.state_dict(), os.path.join(ckpt_dir, "mm_projector.bin"))

    # Save LLM LoRA weights
    unwrapped.language_model.save_pretrained(ckpt_dir)

    # Save optimizer and scheduler states
    torch.save(optimizer.state_dict(), os.path.join(ckpt_dir, "optimizer.bin"))
    torch.save(lr_scheduler.state_dict(), os.path.join(ckpt_dir, "scheduler.bin"))

    if repo_id:
        print(f"\nStep {step}: pushing checkpoint to {repo_id}...")
        try:
            upload_folder(
                folder_path=ckpt_dir,
                repo_id=repo_id,
                token=token,
                path_in_repo=f"checkpoint-{step}",
                commit_message=f"SFT checkpoint step {step}",
                ignore_patterns=["*.lock"],
            )
            print("Checkpoint pushed successfully!")
        except Exception as e:
            print(f"WARNING: HF push failed: {e}")

global_step = 0
start_step = 0

# Adjust start step if resuming
if _resume_ckpt_dir:
    start_step = int(_resume_ckpt_dir.split("-")[-1])
    global_step = start_step

model.train()
completed_epochs = start_step // (len(dataloader) // GRAD_ACCUM_STEPS) if GRAD_ACCUM_STEPS > 0 else 0

for epoch in range(completed_epochs, NUM_EPOCHS):
    if hasattr(batch_sampler, "set_epoch"):
        batch_sampler.set_epoch(epoch)
    epoch_loss = 0.0
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"Epoch {epoch + 1}")

    for step, batch in pbar:
        with accelerator.accumulate(model):
            outputs = model(**batch)
            loss = outputs.loss

            accelerator.backward(loss)
            optimizer.step()
            if accelerator.sync_gradients:
                lr_scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        if accelerator.sync_gradients:
            global_step += 1
            loss_value = loss.detach().float().item()
            epoch_loss += loss_value

            lr = lr_scheduler.get_last_lr()[0]
            samples_per_sec = BATCH_SIZE * GRAD_ACCUM_STEPS

            pbar.set_postfix(
                loss=f"{loss_value:.4f}",
                lr=f"{lr:.2e}",
            )

            # Logs every actual optimizer update.
            accelerator.log(
                {
                    "loss": loss_value,
                    "lr": lr,
                    "epoch": epoch + 1,
                },
                step=global_step,
            )

            if global_step % SAVE_STEPS == 0:
                save_and_push_checkpoint(
                    model, optimizer, lr_scheduler,
                    global_step, OUTPUT_DIR, CHECKPOINT_REPO, HF_TOKEN,
                )

    avg_epoch_loss = epoch_loss / max(len(dataloader), 1)
    accelerator.log({"avg_epoch_loss": avg_epoch_loss}, step=global_step)
    print(f"Epoch {epoch + 1} complete | Avg loss: {avg_epoch_loss:.4f}")

# Save final checkpoint at the end of training
print("Saving final weights...")
save_and_push_checkpoint(model, optimizer, lr_scheduler, global_step, OUTPUT_DIR, CHECKPOINT_REPO, HF_TOKEN)
print(">> Training complete!")

Epoch 1:   0%|          | 0/2667 [00:00<?, ?it/s]


Step 250: pushing checkpoint to abhinav0231/Lily-1.5b-SFT-checkpoints...
Checkpoint pushed successfully!

Step 500: pushing checkpoint to abhinav0231/Lily-1.5b-SFT-checkpoints...
Checkpoint pushed successfully!

Step 750: pushing checkpoint to abhinav0231/Lily-1.5b-SFT-checkpoints...
Checkpoint pushed successfully!

Step 1000: pushing checkpoint to abhinav0231/Lily-1.5b-SFT-checkpoints...
Checkpoint pushed successfully!

Step 1250: pushing checkpoint to abhinav0231/Lily-1.5b-SFT-checkpoints...
Checkpoint pushed successfully!
Epoch 1 complete | Avg loss: 0.3946
Saving final weights...

Step 1334: pushing checkpoint to abhinav0231/Lily-1.5b-SFT-checkpoints...
Checkpoint pushed successfully!
>> Training complete!


## Adapter Merge & Merge Push to HF Hub

This final cell runs after training to perform a lossless merge of the LoRA adapters into the base language model in full precision, then pushes the merged LLM and the projector weights to your new repository.

In [11]:
# Restore standard push_to_hub methods to bypass Unsloth's monkey patch
transformers.PreTrainedModel.push_to_hub = PushToHubMixin.push_to_hub

# 1. Load the tokenizer and add the <image> token first
print("Loading and preparing tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
if "<image>" not in tokenizer.get_vocab():
    tokenizer.add_tokens(["<image>"], special_tokens=True)

# 2. Load the base language model in full precision (unquantized)
print("Loading base language model...")
base_model, _ = FastLanguageModel.from_pretrained(
    model_name     = LLM_MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = DTYPE,
    load_in_4bit   = False,
)

# 3. Resize embeddings to match the checkpoint shape
print(f"Resizing token embeddings to {len(tokenizer)} to match checkpoint shape...")
base_model.resize_token_embeddings(len(tokenizer))

# 4. Load the PEFT adapter checkpoint
checkpoint_dirs = glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*"))
if not checkpoint_dirs:
    raise ValueError(f"No checkpoint directories found in {OUTPUT_DIR}")
checkpoint_path = sorted(checkpoint_dirs, key=lambda x: int(x.split("-")[-1]))[-1]
print(f"Loading trained LoRA adapters from latest local checkpoint: {checkpoint_path}...")
merged_llm = PeftModel.from_pretrained(
    base_model,
    checkpoint_path,
    torch_dtype=DTYPE,
)

# 5. Save merged model and tokenizer locally
print("Merging LoRA adapters into base LLM (lossless 16-bit)...")
final_merged_model = merged_llm.merge_and_unload()

temp_save_dir = "./temp_merged_model"
print(f"Saving merged model locally to {temp_save_dir}...")
final_merged_model.save_pretrained(temp_save_dir, safe_serialization=True)
tokenizer.save_pretrained(temp_save_dir)

# Copy projector weights into the same local folder
projector_path = os.path.join(checkpoint_path, "mm_projector.bin")
shutil.copy(projector_path, os.path.join(temp_save_dir, "mm_projector_final.bin"))

# 6. Upload the entire folder to Hugging Face Hub using standard HfApi
print(f"Uploading entire model folder to Hugging Face: {HF_MERGED_REPO}...")
api = HfApi()
api.create_repo(HF_MERGED_REPO, token=HF_TOKEN, exist_ok=True, private=False)
api.upload_folder(
    folder_path=temp_save_dir,
    repo_id=HF_MERGED_REPO,
    token=HF_TOKEN,
    commit_message="Upload complete merged VLM (LLM + Projector)",
)
print(f">> Complete Multimodal SFT Model uploaded to: https://huggingface.co/{HF_MERGED_REPO}")

Loading and preparing tokenizer...
Loading base language model...
==((====))==  Unsloth 2026.7.5: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu129. CUDA: 8.0. CUDA Toolkit: 12.9. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Resizing token embeddings to 151667 to match checkpoint shape...
Loading trained LoRA adapters from latest local checkpoint: ./sft_output/checkpoint-1334...
Merging LoRA adapters into base LLM (lossless 16-bit)...
Saving merged model locally to ./temp_merged_model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Uploading entire model folder to Hugging Face: abhinav0231/Lily-1.5b-SFT-siglip2...
>> Complete Multimodal SFT Model uploaded to: https://huggingface.co/abhinav0231/Lily-1.5b-SFT-siglip2
